# Cleaning `accounts` data

In [1]:
# in terminal: uv sync
import pandas as pd
import numpy as np
import re

# looading accounts dataset

accounts = pd.read_csv('../../data/raw/HI-Small_accounts.csv')

accounts.head()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


In [2]:
print(accounts['Bank Name'].unique())

<StringArray>
[        'Portugal Bank #4507',             'Canada Bank #27',
                 'UK Bank #33',          'Germany Bank #4815',
 'National Bank of Harrisburg',             'Spain Bank #439',
       'Savings Bank of Omaha',             'Brazil Bank #39',
             'Mexico Bank #16',             'Russia Bank #39',
 ...
      'Switzerland Bank #1397',               'UK Bank #1117',
           'Israel Bank #1124',            'Crytpo Bank #904',
             'Japan Bank #681',            'Brazil Bank #218',
           'Greece Bank #2917',            'Brazil Bank #411',
             'Japan Bank #701',                'UK Bank #483']
Length: 20053, dtype: str


In [3]:
print(accounts['Entity Name'].unique())

<StringArray>
['Sole Proprietorship #50438',         'Corporation #33520',
         'Partnership #35397',         'Corporation #48813',
           'Corporation #889', 'Sole Proprietorship #42987',
 'Sole Proprietorship #20855',         'Partnership #36822',
         'Partnership #36511',         'Partnership #33830',
 ...
          'Corporation #6886',          'Partnership #2830',
 'Sole Proprietorship #16930',          'Corporation #8579',
 'Sole Proprietorship #33971',         'Corporation #40383',
         'Partnership #40687',          'Corporation #1374',
         'Corporation #46077',         'Partnership #49800']
Length: 166207, dtype: str


In [4]:
accounts['Entity Type'] = accounts['Entity Name'].str.extract(r'^([A-Za-z ]+?) #')

## Bank classification

With this section the goal was to use python and regex to work on the bank names in order to extrapolate their locations. <br> Unfortunately, due to this being syntetic data I couldn't get very far with a code based approach and had to get creative and manually modify some of the features and research approaches to cathegorize the banks.

In [5]:
# creating a new dataframe for mapping banks
bank_map = accounts[['Bank Name']].drop_duplicates().reset_index(drop=True)

### Location

In [6]:
def find_country(i):
    if re.search(r'Bank #\d+$', i):
        match = re.search(r'^([A-Za-z ]+?) Bank', i)
        if match:
            return match.group(1) 
    
    if re.search(r'Bank of', i):
        match = re.search(r'Bank of ([A-Za-z ]+?)$', i)
        if match:
            return match.group(1) 

    return 'other'

In [7]:
bank_map['Country'] = bank_map['Bank Name'].apply(find_country)

bank_map['Country'] = bank_map['Country'].replace('Crytpo', 'Crypto') #fixing typo

In [8]:
# city lists to group location by country

usa = [ "Albany", "Arkadelphia", "Augusta", "Billings", "Boston", "Bridgeport", "Butte", "Chicago",  "Cleveland", "Columbus", "Dallas",
        "Danbury", "Denver", "Detroit", "Fairfield", "Fort Wayne", "Harrisburg", "Hartford", "Helena", "Houston", "Huron", "Indianapolis",
        "Lacrosse", "Los Angeles", "Laramie", "Madison", "Miami", "Milford", "New Orleans", "New York", "Newport", "Omaha", "Philadelphia", "Phoenix",
        "Pittsburgh", "Plattsburg", "Portland", "Providence", "Sacramento", "Seattle", "Springfield", "Tampa", "the East", "the North",
        "the South", "the Valley", "the West", "Topeka", "Tuscon", "Watertown"
        ]
uk = ["Lincoln", "Newbury", "Portsmouth"]
can = ["Montpelier"]

bank_map['Country'] = bank_map['Country'].replace(usa, 'USA')
bank_map['Country'] = bank_map['Country'].replace(uk, 'UK')
bank_map['Country'] = bank_map['Country'].replace(can, 'Canada')

#### Coordinates

In [9]:
# extrapolating unique country names to get latitude longitude locations

countries = bank_map[['Country']].drop_duplicates().reset_index(drop=True)
countries = countries[~countries['Country'].isin(['Crypto', 'Other'])]

In [10]:
from geopy.geocoders import Nominatim
import time

In [11]:
geolocator = Nominatim(user_agent="country-coords-script")

# get coordinates
def get_coords(country):
    try:
        time.sleep(1)  # respect rate limit
        location = geolocator.geocode(country)
        if location:
            return pd.Series([location.latitude, location.longitude])
    except:
        pass
    return pd.Series([None, None])

countries[['Latitude', 'Longitude']] = countries['Country'].apply(get_coords)

bank_map = bank_map.merge(countries, on='Country', how='left')

### Types

In [12]:
def find_type(name):
    n = name.lower()
    
    if 'credit union' in n or 'cooperative' in n:
        return 'Cooperative'
    elif 'bancorp' in n:
        return 'Holding'
    elif 'trust' in n:
        return 'Trust'
    elif 'savings' in n or 'thrift' in n:
        return 'Savings'
    elif 'bank' in n:
        return 'Commercial'
    else:
        return None

In [13]:
bank_map['Bank Type'] = bank_map['Bank Name'].apply(find_type)

In [16]:
accounts = accounts.merge(bank_map, on='Bank Name', how='left')

## Feature engineering

In [17]:
accounts.head()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name,Entity Type,Country,Latitude,Longitude,Bank Type
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438,Sole Proprietorship,Portugal,39.662165,-8.135352,Commercial
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520,Corporation,Canada,61.066692,-107.991707,Commercial
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397,Partnership,UK,19.703182,-79.917463,Commercial
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813,Corporation,Germany,51.163818,10.447831,Commercial
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889,Corporation,USA,39.783730,-100.445882,Commercial
